# 02d — Free-text: Supervised Training

Free-text mode, supervised calibration (AUC thresholds + trained weights).
See `02_risk_score_guide.ipynb` for full parameter documentation.

## Setup

Imports → connect → load credentials → open session. See guide for session lifecycle rules.

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
# Run once at the start of every session (fresh or resumed).
import json, sys, os


#!pip install --upgrade git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git

from amygda_ops_risk_score import OpsRiskClient, SessionConfig
from amygda_ops_risk_score import helpers

print("SDK imported successfully.")

In [ ]:
# ── Paths — edit once ────────────────────────────────────────────────────────
API_KEY                = "xxx"  
ARTIFACT_DIR_LABELLING = "artifacts/free_text/labelling/"   # from 01b_labelling_free_text.ipynb
with open(os.path.join(ARTIFACT_DIR_LABELLING, "run_classification.json"), "r") as f:
    labelling_output = json.load(f)
ZIP_PATH               = labelling_output["zip_path"]
ARTIFACT_DIR           = "artifacts/free_text/supervised/risk_score/"  # risk score artifacts land here

import os, json
from amygda_ops_risk_score import OpsRiskClient, SessionConfig
from amygda_ops_risk_score import helpers

client = OpsRiskClient()
client.wait_until_ready()
print(f"Zip: {ZIP_PATH}")

In [ ]:
config  = SessionConfig(name="risk-score-run")
session = client.open_session(
    api_key      = API_KEY,
    config       = config,
    artifact_dir = ARTIFACT_DIR,
)
session.import_model(ZIP_PATH)
print("Session ready. Auth ID:", session.auth_id)


In [ ]:
# ── Kernel-restart recovery — paste values from above ────────────────────────
# AUTH_ID      = "paste-auth-id-here"
# ARTIFACT_DIR = "artifacts/free_text/risk_score/"
#
# session = client.get_session(AUTH_ID, artifact_dir=ARTIFACT_DIR)
# session.status()

## Step 9 — `configure_training`

Upload training log file and configure supervised calibration parameters.
Free-text: `rolling_feature_type` supports `'sum'` and `'flag'` only.

In [ ]:
# ── Training file ────────────────────────────────────────────────────────────
TRAINING_FILE           = "sample_data/free_text_training.csv"
ASSET_ID_COLUMN         = "asset_id"
TIMESTAMP_COLUMN        = "date"
DATE_FORMAT             = "%d/%m/%Y"  # 'infer' auto-detects
ROLLING_WINDOW          = 7
ROLLING_FEATURE_TYPE    = "sum"       # free-text: 'sum'|'flag'  fixed-log: adds 'ewm'|'all'
QUANTILE_FOR_THRESHOLDS = 0.99        # top 1 % of training activity → elevated risk
TRAINING_SHEET          = None        # XLSX only; None for CSV

# ── Failures file (supervised only) ──────────────────────────────────────────
FAILURES_FILE           = "sample_data/free_text_failures.csv"
FAILURE_DATE_COLUMN     = "failure_date"
FAILURE_DATE_FORMAT     = "%d/%m/%Y"  # 'infer' or explicit strftime
PREDICTION_HORIZON_DAYS = 21   # days before failure → labelled positive
EXCLUSION_DAYS_AFTER    = 7    # days after failure excluded (recovery period)
TARGET_IMBALANCE_RATIO  = 10.0 # negatives:positives cap; None keeps all negatives
SAMPLING_STRATEGY       = "block"  # 'block' (temporal) | 'random'
BLOCK_SIZE_DAYS         = 14
N_CANDIDATES            = 50  # threshold grid points per subsystem
FALLBACK_QUANTILE       = 0.99 # quantile used when subsystem has too few positives
MIN_POSITIVES           = 3    # minimum positive rows to attempt AUC training
LR_C                    = 0.5  # logistic regression regularisation

training_config_result = session.configure_training(
    file_path               = TRAINING_FILE,
    asset_id_column         = ASSET_ID_COLUMN,
    timestamp_column        = TIMESTAMP_COLUMN,
    date_format             = DATE_FORMAT,
    rolling_window          = ROLLING_WINDOW,
    rolling_feature_type    = ROLLING_FEATURE_TYPE,
    quantile_for_thresholds = QUANTILE_FOR_THRESHOLDS,
    sheet_name              = TRAINING_SHEET,
    supervised              = True,
    failures_path           = FAILURES_FILE,
    failure_date_column     = FAILURE_DATE_COLUMN,
    failure_date_format     = FAILURE_DATE_FORMAT,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    exclusion_days_after    = EXCLUSION_DAYS_AFTER,
    target_imbalance_ratio  = TARGET_IMBALANCE_RATIO,
    sampling_strategy       = SAMPLING_STRATEGY,
    block_size_days         = BLOCK_SIZE_DAYS,
    n_candidates            = N_CANDIDATES,
    fallback_quantile       = FALLBACK_QUANTILE,
    min_positives           = MIN_POSITIVES,
    lr_c                    = LR_C,
)
print(json.dumps(training_config_result, indent=2))


## Step 10 — `train_risk_model`

Run supervised training on the server (~2–15 min). Artifacts downloaded to `artifact_dir` automatically.

In [ ]:
training_result = session.train_risk_model(timeout=3600)
print(json.dumps({k: v for k, v in training_result.items() if 'path' not in k}, indent=2))


In [ ]:
# ── Supervised training report ───────────────────────────────────────────────
report_path  = training_result.get("supervised_report_path")
weights_path = training_result.get("trained_weights_path")

if report_path:
    with open(report_path) as f:
        report = json.load(f)
    ev = report["evaluation"]
    print(f"Failures used:       {report['n_failures_used']}")
    print(f"Positive rows:       {report['n_positive_rows']} / {report['n_total_rows']} total")
    print(f"Subsystems trained:  {report['subsystems_trained']}")
    print()
    print(f"Baseline  AUC-ROC:        {ev.get('baseline_auc_roc', 'N/A'):.4f}")
    print(f"Supervised AUC-ROC:       {ev.get('supervised_auc_roc', 'N/A'):.4f}")
    print(f"Baseline  Avg Precision:  {ev.get('baseline_avg_precision', 'N/A'):.4f}")
    print(f"Supervised Avg Precision: {ev.get('supervised_avg_precision', 'N/A'):.4f}")


### Supervised training visualisations

Inspect label zones, compare baseline vs supervised metrics, and compare expert vs trained weights.

In [ ]:
# Label timeline: per-asset scatter coloured by label zone
#   Blue = negative (normal)  |  Red = positive (pre-failure)  |  Grey = excluded  |  Stars = failures
helpers.plot_label_timeline(
    training_result["training_fe_path"],
    FAILURES_FILE,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    exclusion_days_after    = EXCLUSION_DAYS_AFTER,
    failure_date_column     = FAILURE_DATE_COLUMN,
)


In [ ]:
# Label distribution over time — positive vs negative row counts per week
# Shows whether positive labels are evenly spread or clustered in one period
helpers.plot_label_distribution(
    training_result["training_fe_path"],
    FAILURES_FILE,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    exclusion_days_after    = EXCLUSION_DAYS_AFTER,
    failure_date_column     = FAILURE_DATE_COLUMN,
    freq = "W",   # 'W' weekly | 'M' monthly
)


In [ ]:
# Metric comparison: baseline (unsupervised quantile + expert weights)
#                     vs supervised (AUC thresholds + trained weights)
helpers.plot_supervised_comparison(training_result["supervised_report_path"])


In [ ]:
# One-time fix: extract model_config.json from the labelling zip into ARTIFACT_DIR
# (train_risk_model() will do this automatically on future sessions via the new endpoint)
import zipfile, os

os.makedirs(ARTIFACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    entry = next((n for n in zf.namelist() if n.endswith("model_config.json")), None)
    if entry:
        dest = f"{ARTIFACT_DIR}model_config.json"
        with zf.open(entry) as src, open(dest, "wb") as dst:
            dst.write(src.read())
        print(f"Extracted → {dest}")
    else:
        print("model_config.json not found in zip — check ZIP_PATH")

In [ ]:
# Weight comparison: expert labelling weights vs trained supervised weights
#   system=None     → system-level comparison
#   system='name'   → subsystem-level comparison for that system
helpers.plot_weight_comparison(
    f"{ARTIFACT_DIR}/model_config.json",
    training_result["trained_weights_path"],
    system = None,
)


### Training period evaluation

Evaluate model performance against training failures using `training_scores.parquet`.

In [ ]:
training_scores_path = training_result.get("training_scores_path")
print(f"Training scores: {training_scores_path}")


In [ ]:
# Risk time series around each failure event (training period)
# Dashed line = failure date | Shaded = prediction horizon
helpers.plot_risk_around_failures(
    training_scores_path,
    FAILURES_FILE,
    failure_date_column     = FAILURE_DATE_COLUMN,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    window_after_days       = 14,
    max_assets              = 12,
)


In [ ]:
# Distribution of risk scores: pre-failure window vs normal operation
# Prints a summary stats table + box plot
helpers.plot_failure_risk_stats(
    training_scores_path,
    FAILURES_FILE,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    exclusion_days_after    = EXCLUSION_DAYS_AFTER,
    failure_date_column     = FAILURE_DATE_COLUMN,
)


### Training feature visualisations

Sanity-check calibrated thresholds and rolling feature distributions.

In [ ]:
# Convenience shortcuts
thresholds_path   = training_result["thresholds_path"]
training_fe_path  = training_result["training_fe_path"]
logs_mapping_path = training_result.get("logs_mapping_path")  # fixed-log only

# 1. Bar chart: calibrated threshold vs mean and max of training distribution
#    Helps spot thresholds that are too low (near mean) or unreachably high (near max)
helpers.plot_calibration_thresholds(
    thresholds_path,
    training_fe_path,
    #systems=["refrigeration", "electrical", "airflow"],
    systems=None # None = all
)


In [ ]:
# 2. Histogram grid: one panel per subsystem showing the risk-feature distribution
#    The red vertical line is the calibrated threshold
helpers.plot_training_feature_distributions(
    training_fe_path,
    thresholds_path,
    #systems=["refrigeration", "electrical", "airflow"],
    systems=None,  # None = all
)


In [ ]:
# 3. Time series of a subsystem's rolling risk feature for selected assets
#    The dashed horizontal line is the calibrated threshold
INSPECT_SYSTEM  = "refrigeration_circuit"
INSPECT_SUBSYS  = "refrigerant_charge"

helpers.plot_training_feature_over_time(
    training_fe_path,
    thresholds_path,
    asset_id = ['HVAC-01', 'HVAC-03', 'HVAC-07'],
    system   = INSPECT_SYSTEM,
    subsystem = INSPECT_SUBSYS,
)


In [ ]:
# 4. Three-panel pipeline view for one asset/subsystem:
#      Top:    risk feature over time + threshold
#      Middle: rolling features (free-text) or per-log-code rolling values (fixed-log)
#      Bottom: binary presence/absence of events
helpers.plot_subsystem_feature_pipeline(
    training_fe_path,
    thresholds_path,
    asset_id    = "HVAC-01",
    system      = INSPECT_SYSTEM,
    subsystem   = INSPECT_SUBSYS,
    is_free_text = training_config_result["is_free_text"],
    logs_mapping = logs_mapping_path,  # only used when is_free_text=False
)


## Step 11 — `configure_generation`

Upload log file to score. Set `weights_source` to `'supervised'` or `'labelling'`.

In [ ]:
GENERATION_FILE  = "sample_data/free_text_generation.csv"
DATE_FORMAT      = "%d/%m/%Y"
GENERATION_SHEET = None
WEIGHTS_SOURCE   = "supervised"  # 'supervised' | 'labelling'

generation_config_result = session.configure_generation(
    file_path      = GENERATION_FILE,
    date_format    = DATE_FORMAT,
    sheet_name     = GENERATION_SHEET,
    weights_source = WEIGHTS_SOURCE,
)
print(json.dumps(generation_config_result, indent=2))


## Step 12 — `generate_risk_scores` 🔒 ONE-WAY DOOR

Generate per-asset risk scores. Downloads complete model zip to `OUTPUT_DIR`.

In [ ]:
# Run generation — ONE-WAY DOOR: cannot be repeated on this session.
# The complete model zip is downloaded to OUTPUT_DIR automatically.
OUTPUT_DIR = "outputs/free_text/supervised/"

generation_result = session.generate_risk_scores(OUTPUT_DIR, timeout=3600)

parquet_path = generation_result.get("parquet_path")
print(f"Risk scores parquet: {parquet_path}")
print(f"Assets scored:       {generation_result['assets_scored']}")
print(f"Date range:          {generation_result['date_range']}")
print("Session wiped automatically — use the zip in OUTPUT_DIR to restart.")


In [ ]:
# Resolve which weights were used — ensures weighted plots match how scores were generated
_trained_w = training_result.get("trained_weights_path")
WEIGHTS_CONFIG = (
    _trained_w
    if _trained_w and os.path.exists(_trained_w)
    else f"{ARTIFACT_DIR}/model_config.json"
)
print(f"Weights config: {WEIGHTS_CONFIG}")


### Risk score visualisations

Plot fleet-wide rankings, heatmaps, single-asset breakdowns, and log drill-downs.
`classified_logs.parquet` is available for free-text log drill-down.

In [ ]:
# Top-N assets ranked by aggregated operational risk
helpers.plot_asset_risk_ranking(
    parquet_path,
    aggregation = "max",   # "mean" | "max" | "min" | "median"
    top_n       = 20,
    title       = "Asset Operational Risk Scores",
)

# Distribution of risk scores across all records
helpers.plot_risk_score_distribution(parquet_path)


In [ ]:
# Multi-asset heatmap: rows = assets, columns = dates
# metric= accepts "operational_risk" or any "{system}_system_risk" column
print("Available metrics:")
for m in helpers.list_risk_metrics(parquet_path): print(f"  {m}")

helpers.plot_risk_heatmap(
    parquet_path,
    metric    = "operational_risk",
    asset_ids = ['HVAC-01', 'HVAC-03', 'HVAC-07'],  # None = all assets
    title     = "Multi-Asset Operational Risk Heatmap",
)



In [ ]:
# Tile panel for a single date — one operational-risk tile per asset
# with system-level tiles below.  Colour: blue (0–33), yellow (33–66), red (66–100)
SCORE_DATE = "2024-02-15"   # ← edit
helpers.plot_daily_risk_snapshot(
    parquet_path,
    date      = SCORE_DATE,
    min_score = 0.0,           # hide assets below this score
    asset_ids = ['HVAC-01', 'HVAC-03', 'HVAC-07'],
)


In [ ]:
# Single asset — heatmap of all risk types over time
ASSET_TO_INSPECT = "HVAC-01"   # ← edit

# Raw scores
helpers.plot_asset_risk_over_time(
    parquet_path, asset_id=ASSET_TO_INSPECT,
    title=f"Risk Analysis — {ASSET_TO_INSPECT}",
)

# Weighted view (system rows scaled by their model weight)
helpers.plot_asset_risk_over_time(
    parquet_path, asset_id=ASSET_TO_INSPECT,
    weighted=True, model_config=WEIGHTS_CONFIG,
    title=f"Risk Analysis — Weighted — {ASSET_TO_INSPECT}",
)



In [ ]:
# Stacked weighted system contributions + operational risk line
helpers.plot_system_contributions(
    parquet_path,
    asset_id     = ASSET_TO_INSPECT,
    model_config = WEIGHTS_CONFIG,
)


In [ ]:
# Drill into one system: subsystem heatmap over time
DRILL_SYSTEM = INSPECT_SYSTEM   # ← edit

helpers.plot_asset_system_over_time(
    parquet_path, asset_id=ASSET_TO_INSPECT,
    system=DRILL_SYSTEM, title=f"{DRILL_SYSTEM} — {ASSET_TO_INSPECT}",
)
helpers.plot_asset_system_over_time(
    parquet_path, asset_id=ASSET_TO_INSPECT,
    system=DRILL_SYSTEM, weighted=True, model_config=WEIGHTS_CONFIG,
    title=f"{DRILL_SYSTEM} — Weighted — {ASSET_TO_INSPECT}",
)


In [ ]:
# Log-level drill-down: rolling features + binary presence + log counts
# for a specific asset / system / subsystem window
#
# Free-text mode  → source = classified_logs.parquet
# Fixed-log mode  → source = risk_scores.parquet + logs_mapping
import os
ASSET_ID   = "HVAC-01"
SYSTEM     = INSPECT_SYSTEM
SUBSYSTEM  = INSPECT_SUBSYS
END_DATE   = "2024-02-15"
DAYS_BACK  = 60
LOG_COLUMN = "maintenance_log"

classified_logs_path = generation_result.get("classified_logs_path")
logs_mapping_path    = generation_result.get("logs_mapping_path")

if classified_logs_path and os.path.exists(classified_logs_path):
    # Free-text mode
    helpers.plot_subsystem_log_activity(
        classified_logs_path,
        asset_id=ASSET_ID, system=SYSTEM, subsystem=SUBSYSTEM,
        date=END_DATE, days_back=DAYS_BACK, log_column=LOG_COLUMN,
        risk_source=parquet_path,
    )
elif parquet_path and logs_mapping_path:
    # Fixed-log mode
    helpers.plot_subsystem_log_activity(
        parquet_path,
        asset_id=ASSET_ID, system=SYSTEM, subsystem=SUBSYSTEM,
        date=END_DATE, days_back=DAYS_BACK, log_column=LOG_COLUMN,
        risk_source=parquet_path, logs_mapping=logs_mapping_path, is_free_text=False,
    )


In [ ]:
if not os.path.exists(classified_logs_path):
    print("classified_logs.parquet not available — this helper is for free-text mode only.")
    print(f"Expected at: {classified_logs_path}")
else:
    logs_df = helpers.get_logs_for_subsystem(
        classified_logs_path,
        asset_id=ASSET_ID,
        system=SYSTEM,
        subsystem=SUBSYSTEM,
        date=END_DATE,
        days_back=DAYS_BACK,
        log_column=LOG_COLUMN,
    )
    print(f"{len(logs_df)} log entries found")
    display(logs_df)

### Failure performance

Evaluate risk scores against failure events in the generation period.

In [ ]:
helpers.plot_risk_around_failures(
    parquet_path,
    FAILURES_FILE,
    failure_date_column     = FAILURE_DATE_COLUMN,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    window_after_days       = 14,
    max_assets              = 6,
)


In [ ]:
helpers.plot_failure_risk_stats(
    parquet_path,
    FAILURES_FILE,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    exclusion_days_after    = EXCLUSION_DAYS_AFTER,
    failure_date_column     = FAILURE_DATE_COLUMN,
)
